# FRC Vision Scout — train -> CPU-deployable model (KAGGLE / free T4)

Kaggle version of the engine. Trains a custom FUEL+robot detector from your match clip (no manual
labeling: YOLO11x+SAHI teacher) and exports an **OpenVINO INT8** model that runs **LIVE on your
Intel Core Ultra (CPU/iGPU/NPU), no CUDA, no internet.**

## Setup before running
1. **Settings (right panel) -> Accelerator -> GPU T4 x2** (needs phone-verified account).
2. **Settings -> Internet -> On** (for pip + model weights).
3. **Input -> Add Input -> your dataset** containing `match_720.mp4` (upload it as a Dataset first).
   The notebook auto-finds any .mp4 under `/kaggle/input/`.
4. Run all cells top to bottom.

## Notes
- Outputs go to `/kaggle/working/` (download from the **Output** tab, or **Save Version**).
- `/kaggle/working` is wiped between sessions, so either finish in one run (~45 min) or **Save
  Version** to keep the checkpoint. Training auto-resumes from a checkpoint if one exists.


In [ ]:
#@title 1. Setup + GPU
!pip -q install ultralytics openvino nncf supervision sahi
import torch, os, ultralytics
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2 (and phone-verify your account)'
print('ultralytics', ultralytics.__version__, '| GPU', torch.cuda.get_device_name(0))
ROOT='/kaggle/working/frc_vision'; os.makedirs(ROOT, exist_ok=True)
DS=f'{ROOT}/dataset'; os.makedirs(DS, exist_ok=True)
print('workdir:', ROOT)

In [ ]:
#@title 2. Locate the match clip (robust readability check)
import glob, cv2
cands=sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True))
assert cands, 'No .mp4 in /kaggle/input — Add Input -> your frc-match dataset'
MP4=cands[0]
c=cv2.VideoCapture(MP4)
ok,_=c.read()                       # actually decode a frame (FRAME_COUNT metadata is unreliable)
W=int(c.get(3)); H=int(c.get(4)); fps=c.get(5); n=int(c.get(cv2.CAP_PROP_FRAME_COUNT))
c.release()
print('using clip:', MP4)
print(f'{W}x{H} @ {round(fps,1)} fps | frame_count_meta={n} | first-frame readable={ok}')
assert ok and W>0, 'clip is genuinely unreadable — re-upload the dataset'

In [ ]:
#@title 3. Auto-label (teacher ensemble: YOLO11x+SAHI for balls, YOLO-World for robots)
# Pseudo-labels, no human. 2 classes -> 0=ball, 1=robot. Writes YOLO dataset to /kaggle/working.
import cv2, random, numpy as np, os
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from ultralytics import YOLO
random.seed(0)
for s in ['images/train','images/val','labels/train','labels/val']: os.makedirs(f'{DS}/{s}',exist_ok=True)
ball_teacher=AutoDetectionModel.from_pretrained(model_type='ultralytics',model_path='yolo11x.pt',confidence_threshold=0.05,device='cuda:0')
world=YOLO('yolov8x-worldv2.pt'); world.set_classes(['robot'])
cap=cv2.VideoCapture(MP4); fps=cap.get(5); total=int(cap.get(7)); W=int(cap.get(3)); H=int(cap.get(4))
PLAY=(0.125,0.62)   # keep air column + carpet + front; drop scoreboard/reflection
step=max(1,int(fps*0.5))   # ~2 frames/sec sampled for labeling
idx=n=0
while True:
    ok=cap.grab()
    if not ok: break
    if idx%step==0:
        ok2,f=cap.retrieve()
        if not ok2: break
        cv2.imwrite('f.png',f)
        sp=get_sliced_prediction('f.png',ball_teacher,slice_height=384,slice_width=384,overlap_height_ratio=0.25,overlap_width_ratio=0.25,verbose=0)
        lines=[]
        for o in sp.object_prediction_list:
            if o.category.id!=32: continue
            b=o.bbox; cx=(b.minx+b.maxx)/2; cy=(b.miny+b.maxy)/2
            if not (PLAY[0]*H<=cy<=PLAY[1]*H): continue
            lines.append(f'0 {cx/W:.6f} {cy/H:.6f} {(b.maxx-b.minx)/W:.6f} {(b.maxy-b.miny)/H:.6f}')
        for box in world.predict(f,conf=0.05,imgsz=1280,verbose=False)[0].boxes.xyxy.cpu().numpy():
            x1,y1,x2,y2=box; cy=(y1+y2)/2
            if not (PLAY[0]*H<=cy<=PLAY[1]*H): continue
            lines.append(f'1 {((x1+x2)/2)/W:.6f} {cy/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}')
        sub='val' if random.random()<0.15 else 'train'
        cv2.imwrite(f'{DS}/images/{sub}/{n:05d}.png',f)
        open(f'{DS}/labels/{sub}/{n:05d}.txt','w').write(chr(10).join(lines))
        n+=1
        if n%20==0: print('labeled',n,'frames')
    idx+=1
cap.release()
open(f'{DS}/data.yaml','w').write(f"path: {DS}\ntrain: images/train\nval: images/val\nnames:\n  0: ball\n  1: robot\n")
print('AUTO-LABEL done:',n,'frames ->',DS)

In [ ]:
#@title 4. Train the specialist student (YOLO11s) — AUTO-RESUMES if re-run
from ultralytics import YOLO
import os
ckpt=f'{ROOT}/runs/fuel_robot/weights/last.pt'
if os.path.exists(ckpt):
    print('Found checkpoint -> RESUMING from', ckpt)
    m=YOLO(ckpt); m.train(resume=True)
else:
    print('Fresh training run')
    m=YOLO('yolo11s.pt')
    m.train(data=f'{DS}/data.yaml', epochs=120, imgsz=1024, batch=16, device=0,
            project=f'{ROOT}/runs', name='fuel_robot', exist_ok=True,
            mosaic=1.0, close_mosaic=15, copy_paste=0.3, hsv_v=0.5, hsv_s=0.5,
            degrees=5, translate=0.1, scale=0.5, fliplr=0.5, erasing=0.3,
            patience=40, cache=False)
BEST=f'{ROOT}/runs/fuel_robot/weights/best.pt'
print('trained ->',BEST)

In [ ]:
#@title 5. Export OpenVINO INT8 (CUDA-free deploy artifact) + ONNX
from ultralytics import YOLO
import glob
BEST=f'{ROOT}/runs/fuel_robot/weights/best.pt'
DEPLOY_IMGSZ=768          # 640 = faster, 960 = higher small-ball recall. Must match live script.
m=YOLO(BEST)
ov=m.export(format='openvino', int8=True, data=f'{DS}/data.yaml', imgsz=DEPLOY_IMGSZ, half=False)
print('OpenVINO INT8 ->', ov)
onnx=m.export(format='onnx', imgsz=DEPLOY_IMGSZ, opset=13, simplify=True)
print('ONNX ->', onnx)
imgs=glob.glob(f'{DS}/images/val/*.png') or glob.glob(f'{DS}/images/train/*.png')
if imgs:
    r=YOLO(ov, task='detect').predict(imgs[0], imgsz=DEPLOY_IMGSZ, conf=0.15, verbose=False)[0]
    print('INT8 model OK — detections on a sample frame:', len(r.boxes))

In [ ]:
#@title 6. (Optional) Track the full match -> annotated demo video
from ultralytics import YOLO
import cv2, numpy as np, json, glob, subprocess
MP4=(sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True)) or [''])[0]
BEST=f'{ROOT}/runs/fuel_robot/weights/best.pt'
m=YOLO(BEST)
res=m.track(source=MP4, imgsz=1280, conf=0.15, iou=0.5, tracker='botsort.yaml', persist=True, stream=True, verbose=False)
cap=cv2.VideoCapture(MP4); fps=cap.get(5); W=int(cap.get(3)); H=int(cap.get(4)); cap.release()
vw=cv2.VideoWriter('out.mp4',cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
COL={0:(0,255,255),1:(255,90,0)}; per_frame=[]
for r in res:
    f=r.orig_img.copy(); nb=nr=0
    if r.boxes is not None and len(r.boxes):
        ids=r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None else [-1]*len(r.boxes)
        for b,c,tid in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int), ids):
            x1,y1,x2,y2=b.astype(int)
            if c==0: cv2.circle(f,((x1+x2)//2,(y1+y2)//2),max(3,(x2-x1)//2),COL[0],2); nb+=1
            else: cv2.rectangle(f,(x1,y1),(x2,y2),COL[1],2); cv2.putText(f,f'R{tid}',(x1,y1-3),cv2.FONT_HERSHEY_SIMPLEX,0.5,COL[1],1); nr+=1
    cv2.putText(f,f'FUEL {nb}  ROBOTS {nr}',(8,28),cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,0,255),2)
    vw.write(f); per_frame.append({'balls':nb,'robots':nr})
vw.release()
subprocess.run('ffmpeg -hide_banner -loglevel error -y -i out.mp4 -c:v libx264 -pix_fmt yuv420p /kaggle/working/tracked_custom.mp4',shell=True)
json.dump(per_frame,open(f'{ROOT}/tracks.json','w'))
print('mean balls/frame',round(np.mean([p['balls'] for p in per_frame]),1),'mean robots/frame',round(np.mean([p['robots'] for p in per_frame]),1))
print('saved /kaggle/working/tracked_custom.mp4')

In [ ]:
#@title 7. Package the DEPLOY model + print mAP (download from the Output tab)
import shutil, os
from ultralytics import YOLO
run=f'{ROOT}/runs/fuel_robot'; wdir=f'{run}/weights'
ovdirs=[d for d in os.listdir(wdir) if d.endswith('openvino_model')]
assert ovdirs, 'No OpenVINO export found — run cell 5 first.'
shutil.make_archive('/kaggle/working/frc_model_openvino', 'zip', f'{wdir}/{ovdirs[0]}')
print('packaged', ovdirs[0], '-> /kaggle/working/frc_model_openvino.zip')
try: print('val mAP50-95:', YOLO(f'{wdir}/best.pt').val(data=f'{DS}/data.yaml').box.map)
except Exception as e: print('val skipped:', str(e)[:80])
print('\nDOWNLOAD: open the right-side Output tab (or Save Version) and grab')
print('  /kaggle/working/frc_model_openvino.zip  (your live model)')
print('Then on your laptop:  python src/live_openvino.py <unzipped_dir> --device intel:gpu --source match_720.mp4 --save out.mp4')